# Đề tài: Gom cụm văn bản báo chí tiếng Việt bằng biểu diễn trọng số từ, gom cụm và khai thác mẫu

## Bối cảnh
Sự gia tăng nhanh của dữ liệu báo chí tiếng Việt trên các nền tảng số đặt ra nhu cầu tổ chức, khám phá và phân nhóm văn bản một cách tự động. Tuy nhiên, văn bản tiếng Việt có nhiều đặc thù như từ ghép đa âm tiết, tính đa nghĩa và mức độ chồng lấn chủ đề giữa các chuyên mục, khiến bài toán gom cụm trở nên khó hơn so với dữ liệu có cấu trúc.

## Mục tiêu
Đồ án này nhằm xây dựng một pipeline gom cụm văn bản tiếng Việt theo hướng đầy đủ và có hệ thống, bao gồm:
- gộp dữ liệu thành một file chính,
- tiền xử lý văn bản chi tiết,
- tách từ tiếng Việt và loại bỏ từ dừng,
- biểu diễn văn bản bằng trọng số từ,
- thực hiện gom cụm,
- đánh giá chất lượng cụm bằng các chỉ số phù hợp,
- và khai thác mẫu đặc trưng trong từng cụm để hỗ trợ diễn giải kết quả.

## Ý nghĩa học thuật
Đề tài giúp làm rõ vai trò của các bước tiền xử lý tiếng Việt, biểu diễn văn bản và lựa chọn thuật toán trong bài toán gom cụm. Ngoài ra, việc kết hợp giữa đánh giá cụm và khai thác mẫu còn giúp tăng khả năng giải thích cụm, thay vì chỉ dừng ở các chỉ số định lượng.

## Đầu ra mong đợi
Sau notebook này, đầu ra mong đợi gồm:
- một file dữ liệu chính đã được gộp và kiểm tra,
- một phiên bản dữ liệu đã qua tiền xử lý để phục vụ gom cụm,
- các đặc trưng văn bản phù hợp cho bước mô hình hóa,
- và cơ sở rõ ràng để chuyển sang bước gom cụm, đánh giá và phân tích cụm trong các notebook tiếp theo.

In [3]:
# Cell 2: import thư viện và khai báo đường dẫn dữ liệu
# Cell này dùng để chuẩn bị môi trường làm việc, xác định file đầu vào và nơi lưu file đầu ra.

import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("D:/BAI TEST/data/tuoitre")
FILES = [
    "cong_nghe.csv",
    "du_lich.csv",
    "giao_duc.csv",
    "suc_khoe.csv",
    "the_thao.csv",
    "xe.csv",
]

OUTPUT_DIR = Path("D:/BAI TEST/data/processed_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_MAIN = OUTPUT_DIR / "bai_test_do_an_main.csv"

print("Danh sách file đầu vào:")
for file_name in FILES:
    file_path = DATA_DIR / file_name
    print("-", file_path, "| tồn tại:", file_path.exists())

print("\nThư mục đầu ra:")
print(OUTPUT_DIR)

print("\nFile đầu ra chính sẽ lưu tại:")
print(OUTPUT_MAIN)

Danh sách file đầu vào:
- D:\BAI TEST\data\tuoitre\cong_nghe.csv | tồn tại: True
- D:\BAI TEST\data\tuoitre\du_lich.csv | tồn tại: True
- D:\BAI TEST\data\tuoitre\giao_duc.csv | tồn tại: True
- D:\BAI TEST\data\tuoitre\suc_khoe.csv | tồn tại: True
- D:\BAI TEST\data\tuoitre\the_thao.csv | tồn tại: True
- D:\BAI TEST\data\tuoitre\xe.csv | tồn tại: True

Thư mục đầu ra:
D:\BAI TEST\data\processed_data

File đầu ra chính sẽ lưu tại:
D:\BAI TEST\data\processed_data\bai_test_do_an_main.csv


In [4]:
# Cell 3: đọc và gộp 6 file dữ liệu thành một bảng duy nhất
# Cell này giúp tạo file dữ liệu chính ban đầu để kiểm tra quy mô và phân bố dữ liệu trước khi làm sạch sâu hơn.

df_list = []

for file_name in FILES:
    file_path = DATA_DIR / file_name
    df_temp = pd.read_csv(file_path).copy()
    df_temp["file_name"] = file_name
    df_list.append(df_temp)

df_main = pd.concat(df_list, ignore_index=True)

df_main.insert(0, "doc_id", range(1, len(df_main) + 1))

print("Kích thước dữ liệu sau khi gộp:", df_main.shape)
print("\nCác cột hiện có:")
print(df_main.columns.tolist())

print("\nSố dòng theo file nguồn:")
print(df_main["file_name"].value_counts())

display(df_main.head(3))

Kích thước dữ liệu sau khi gộp: (8830, 9)

Các cột hiện có:
['doc_id', 'source', 'category_clean', 'title', 'description', 'content', 'published_date', 'url', 'file_name']

Số dòng theo file nguồn:
file_name
du_lich.csv      1984
giao_duc.csv     1983
suc_khoe.csv     1976
the_thao.csv     1407
cong_nghe.csv     991
xe.csv            489
Name: count, dtype: int64


,doc_id,source,category_clean,title,description,content,published_date,url,file_name
0,1,tuoitre,cong_nghe,Làn sóng máy tính lượng tử sau cơn sốt AI,"Quý 1-2026, thế giới chứng kiến loạt công ty v...","Đáng chú ý, khác với các giai đoạn sàng lọc cô...",2026-04-11T11:36:52+07:00,https://tuoitre.vn/lan-song-may-tinh-luong-tu-...,cong_nghe.csv
1,2,tuoitre,cong_nghe,Meta đưa AI vào Messenger trả lời khách hàng 24/7,Khách hàng nhắn tin lúc nửa đêm vẫn có người t...,Meta vừa giới thiệu trợ lý kinh doanh Business...,2026-04-11T07:11:28+07:00,https://tuoitre.vn/meta-dua-ai-vao-messenger-t...,cong_nghe.csv
2,3,tuoitre,cong_nghe,Hà Tiên sử dụng trí tuệ nhân tạo phục vụ hành ...,"Ngày 10-4, UBND phường Hà Tiên (An Giang) ra m...","Phát biểu tại buổi lễ ra mắt, ông Trần Hải Quố...",2026-04-10T16:23:50+07:00,https://tuoitre.vn/ha-tien-su-dung-tri-tue-nha...,cong_nghe.csv


In [5]:
# Cell 4: kiểm tra nhanh chất lượng dữ liệu sau khi gộp
# Cell này giúp phát hiện các vấn đề cơ bản như thiếu dữ liệu, trùng URL và trùng nội dung trước khi làm sạch sâu hơn.

print("Số giá trị thiếu theo cột:")
display(df_main.isna().sum().to_frame("missing_count"))

url_duplicated = df_main["url"].duplicated().sum()
title_content_duplicated = df_main.duplicated(subset=["title", "content"]).sum()

print("\nSố URL trùng:", url_duplicated)
print("Số bản ghi trùng theo cặp title + content:", title_content_duplicated)

print("\nPhân bố theo category_clean:")
display(df_main["category_clean"].value_counts().to_frame("count"))

print("\nKhoảng thời gian dữ liệu:")
published_series = pd.to_datetime(df_main["published_date"], errors="coerce")
print("Ngày nhỏ nhất:", published_series.min())
print("Ngày lớn nhất:", published_series.max())

Số giá trị thiếu theo cột:


,missing_count
doc_id,0
source,0
category_clean,0
title,0
description,0
content,0
published_date,0
url,0
file_name,0



Số URL trùng: 101
Số bản ghi trùng theo cặp title + content: 102

Phân bố theo category_clean:


,count
category_clean,
du_lich,1984
giao_duc,1983
suc_khoe,1976
the_thao,1407
cong_nghe,991
xe,489



Khoảng thời gian dữ liệu:
Ngày nhỏ nhất: 2022-06-01 16:12:20+07:00
Ngày lớn nhất: 2026-04-12 22:42:52+07:00


### Nhận xét nhanh sau bước kiểm tra dữ liệu

Dữ liệu sau khi gộp không có giá trị thiếu ở 9 cột, đây là một tín hiệu tốt vì giúp giảm rủi ro khi tiền xử lý văn bản. Tuy nhiên, bộ dữ liệu hiện vẫn còn **101 URL trùng** và **102 bản ghi trùng theo cặp `title + content`**, cho thấy cần có bước loại trùng trước khi xây dựng tập văn bản chính thức.

Ngoài ra, khoảng thời gian dữ liệu trải từ **2022-06-01** đến **2026-04-12**, khá rộng so với các file còn lại. Điều này cho thấy chuyên mục `the_thao` có khả năng chứa các bài cũ hơn đáng kể, và đây là điểm cần lưu ý khi làm sạch hoặc phân tích độ đồng nhất của tập dữ liệu.

In [6]:
# Cell 5: tạo cột văn bản chính và các cột chuẩn hóa tối thiểu để phục vụ lọc trùng
# Cell này giúp chuẩn bị dữ liệu cho bước dedup và làm sạch bằng cách gom nội dung văn bản về một dạng nhất quán hơn.

df_work = df_main.copy()

df_work["raw_text"] = (
    df_work["title"].astype(str).str.strip() + " . " +
    df_work["description"].astype(str).str.strip() + " . " +
    df_work["content"].astype(str).str.strip()
)

for col in ["title", "description", "content", "raw_text"]:
    df_work[f"{col}_norm"] = (
        df_work[col]
        .astype(str)
        .str.lower()
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

print("Kích thước dữ liệu làm việc:", df_work.shape)

show_cols = [
    "doc_id", "category_clean", "title", "description", "content", "raw_text"
]
display(df_work[show_cols].head(2))

Kích thước dữ liệu làm việc: (8830, 14)


,doc_id,category_clean,title,description,content,raw_text
0,1,cong_nghe,Làn sóng máy tính lượng tử sau cơn sốt AI,"Quý 1-2026, thế giới chứng kiến loạt công ty v...","Đáng chú ý, khác với các giai đoạn sàng lọc cô...",Làn sóng máy tính lượng tử sau cơn sốt AI . Qu...
1,2,cong_nghe,Meta đưa AI vào Messenger trả lời khách hàng 24/7,Khách hàng nhắn tin lúc nửa đêm vẫn có người t...,Meta vừa giới thiệu trợ lý kinh doanh Business...,Meta đưa AI vào Messenger trả lời khách hàng 2...


In [7]:
# Cell 6: tính độ dài văn bản để phục vụ lọc bài quá ngắn và kiểm tra nhanh phân bố độ dài
# Cell này giúp xác định mức độ đầy đủ của nội dung trước khi loại trùng và làm sạch sâu hơn.

for col in ["title", "description", "content", "raw_text"]:
    df_work[f"{col}_word_count"] = (
        df_work[col]
        .astype(str)
        .str.split()
        .str.len()
    )

summary_cols = [
    "title_word_count",
    "description_word_count",
    "content_word_count",
    "raw_text_word_count",
]

print("Thống kê độ dài văn bản:")
display(df_work[summary_cols].describe().T)

print("\nSố bài có content dưới 30 từ:")
print((df_work["content_word_count"] < 30).sum())

Thống kê độ dài văn bản:


,count,mean,std,min,25%,50%,75%,max
title_word_count,8830.0,14.903737,3.655548,4.0,12.0,15.0,17.0,26.0
description_word_count,8830.0,37.753114,8.674203,15.0,31.0,37.0,44.0,59.0
content_word_count,8830.0,592.757191,302.067642,23.0,385.0,534.0,737.0,6774.0
raw_text_word_count,8830.0,647.414043,303.757898,82.0,437.0,589.0,795.0,6822.0



Số bài có content dưới 30 từ:
1


### Nhận xét nhanh về độ dài văn bản

Độ dài `content` trung bình khoảng **593 từ**, trung vị khoảng **534 từ**, cho thấy phần lớn bài viết có nội dung tương đối đầy đủ và giàu thông tin. Chỉ có **1 bài dưới 30 từ**, nên bước lọc bài quá ngắn nếu áp dụng sẽ gần như không làm thay đổi quy mô dữ liệu, nhưng vẫn nên giữ để đảm bảo tính nhất quán của pipeline.

In [8]:
# Cell 7: loại trùng theo URL và theo cặp title + content
# Cell này thực hiện bước làm sạch quan trọng để tránh nhiều bản ghi trùng gây nhiễu khi gom cụm.

before_rows = len(df_work)

df_dedup = df_work.drop_duplicates(subset=["url"], keep="first").copy()
after_url_dedup = len(df_dedup)

df_dedup = df_dedup.drop_duplicates(subset=["title_norm", "content_norm"], keep="first").copy()
after_title_content_dedup = len(df_dedup)

print("Số dòng ban đầu:", before_rows)
print("Sau khi loại trùng theo URL:", after_url_dedup)
print("Sau khi loại trùng theo title_norm + content_norm:", after_title_content_dedup)

print("\nSố dòng bị loại do URL trùng:", before_rows - after_url_dedup)
print("Số dòng bị loại thêm do title_norm + content_norm trùng:", after_url_dedup - after_title_content_dedup)

display(df_dedup[["doc_id", "category_clean", "title", "url"]].head(3))

Số dòng ban đầu: 8830
Sau khi loại trùng theo URL: 8729
Sau khi loại trùng theo title_norm + content_norm: 8728

Số dòng bị loại do URL trùng: 101
Số dòng bị loại thêm do title_norm + content_norm trùng: 1


,doc_id,category_clean,title,url
0,1,cong_nghe,Làn sóng máy tính lượng tử sau cơn sốt AI,https://tuoitre.vn/lan-song-may-tinh-luong-tu-...
1,2,cong_nghe,Meta đưa AI vào Messenger trả lời khách hàng 24/7,https://tuoitre.vn/meta-dua-ai-vao-messenger-t...
2,3,cong_nghe,Hà Tiên sử dụng trí tuệ nhân tạo phục vụ hành ...,https://tuoitre.vn/ha-tien-su-dung-tri-tue-nha...


### Nhận xét nhanh sau bước loại trùng

Sau bước loại trùng, bộ dữ liệu giảm từ **8.830** xuống **8.728** bài. Trong đó:
- **101 bài** bị loại do trùng URL,
- **1 bài** bị loại thêm do trùng theo cặp `title_norm + content_norm`.

Điều này cho thấy phần lớn vấn đề trùng lặp đến từ URL, còn mức độ trùng hoàn toàn theo tiêu đề và nội dung là rất thấp sau khi đã loại URL trùng. Bộ dữ liệu sau bước này đã sạch hơn và phù hợp hơn để chuyển sang bước tạo tập văn bản chính thức cho tiền xử lý.

In [9]:
# Cell 8: lọc bài quá ngắn và chốt file dữ liệu sạch
# Cell này giữ lại các bản ghi đạt ngưỡng độ dài, sau đó chỉ lưu những cột thật sự cần thiết cho file dữ liệu chính.

MIN_WORDS = 30

df_final = df_dedup[df_dedup["content_word_count"] >= MIN_WORDS].copy()

print("--- TỔNG KẾT NOTEBOOK 1 ---")
print("Số lượng bài báo ban đầu:", len(df_main))
print("Số lượng bài báo sau loại trùng:", len(df_dedup))
print("Số lượng bài báo sau lọc ngắn:", len(df_final))
print("Tổng cộng đã loại bỏ:", len(df_main) - len(df_final), "bài")

print("\n--- PHÂN BỐ CHUYÊN MỤC CUỐI CÙNG ---")
category_final = df_final["category_clean"].value_counts().to_frame("count")
category_final["percentage"] = (category_final["count"] / len(df_final) * 100).round(2)
display(category_final)

final_cols = [
    "doc_id",
    "source",
    "category_clean",
    "title",
    "description",
    "content",
    "published_date",
    "url",
]

df_final_export = df_final[final_cols].copy()

OUTPUT_MAIN_CLEAN = OUTPUT_DIR / "bai_test_do_an_main_clean.csv"
df_final_export.to_csv(OUTPUT_MAIN_CLEAN, index=False, encoding="utf-8-sig")

print("\nKích thước file sạch cuối cùng:", df_final_export.shape)
print("Đã lưu file sạch tại:")
print(OUTPUT_MAIN_CLEAN)

display(df_final_export.head(3))

--- TỔNG KẾT NOTEBOOK 1 ---
Số lượng bài báo ban đầu: 8830
Số lượng bài báo sau loại trùng: 8728
Số lượng bài báo sau lọc ngắn: 8727
Tổng cộng đã loại bỏ: 103 bài

--- PHÂN BỐ CHUYÊN MỤC CUỐI CÙNG ---


,count,percentage
category_clean,,
du_lich,1979,22.68
giao_duc,1941,22.24
suc_khoe,1929,22.10
the_thao,1402,16.07
cong_nghe,991,11.36
xe,485,5.56



Kích thước file sạch cuối cùng: (8727, 8)
Đã lưu file sạch tại:
D:\BAI TEST\data\processed_data\bai_test_do_an_main_clean.csv


,doc_id,source,category_clean,title,description,content,published_date,url
0,1,tuoitre,cong_nghe,Làn sóng máy tính lượng tử sau cơn sốt AI,"Quý 1-2026, thế giới chứng kiến loạt công ty v...","Đáng chú ý, khác với các giai đoạn sàng lọc cô...",2026-04-11T11:36:52+07:00,https://tuoitre.vn/lan-song-may-tinh-luong-tu-...
1,2,tuoitre,cong_nghe,Meta đưa AI vào Messenger trả lời khách hàng 24/7,Khách hàng nhắn tin lúc nửa đêm vẫn có người t...,Meta vừa giới thiệu trợ lý kinh doanh Business...,2026-04-11T07:11:28+07:00,https://tuoitre.vn/meta-dua-ai-vao-messenger-t...
2,3,tuoitre,cong_nghe,Hà Tiên sử dụng trí tuệ nhân tạo phục vụ hành ...,"Ngày 10-4, UBND phường Hà Tiên (An Giang) ra m...","Phát biểu tại buổi lễ ra mắt, ông Trần Hải Quố...",2026-04-10T16:23:50+07:00,https://tuoitre.vn/ha-tien-su-dung-tri-tue-nha...
